In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import re
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

In [3]:
import pandas as pd
import numpy as np

# Example: create dummy predictions
# Replace `test.csv` with your actual test file if needed

df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(df.shape)
print(test_df.shape)

(2000, 8)
(500, 7)


In [4]:
df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df["prompt"].nunique()

1758

In [7]:
df["answer"].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df["id"].duplicated().sum()

np.int64(0)

In [10]:
df_cols = df.select_dtypes(exclude="number").columns.tolist()

In [11]:
# The columns we want to analyze
text_cols = ['prompt', 'A', 'B', 'C', 'D', 'E']

# List to store our computed statistics
stats = []

for col in text_cols:
    # Convert to string to handle any accidental NaN/null values gracefully
    col_data = df[col].astype(str)
    
    # Calculate lengths
    char_lengths = col_data.str.len()
    word_counts = col_data.str.split().str.len()
    
    # Append the metrics to our list
    stats.append({
        'Column': col,
        'Avg Char Length': round(char_lengths.mean(), 2),
        'Avg Word Count': round(word_counts.mean(), 2),
        'Min Char Length': char_lengths.min(),
        'Max Char Length': char_lengths.max(),
        'Min Word Count': word_counts.min(),
        'Max Word Count': word_counts.max()
    })

# Convert to a DataFrame for a clean, tabular display
eda_df = pd.DataFrame(stats)
print(eda_df.to_markdown(index=False))

| Column   |   Avg Char Length |   Avg Word Count |   Min Char Length |   Max Char Length |   Min Word Count |   Max Word Count |
|:---------|------------------:|-----------------:|------------------:|------------------:|-----------------:|-----------------:|
| prompt   |            117.67 |            18.15 |                19 |               337 |                3 |               51 |
| A        |            164.09 |            26.15 |                 1 |               472 |                1 |               81 |
| B        |            166.62 |            26.52 |                 2 |               662 |                1 |              118 |
| C        |            167.23 |            26.55 |                 2 |               530 |                1 |               82 |
| D        |            162.56 |            26    |                 1 |               450 |                1 |               78 |
| E        |            164.23 |            26.19 |                 2 |               587 

In [12]:
def clean_text(text):
    """
    Base text cleaning function. 
    Designed to be modular so future steps (like regex stripping) can be added here.
    """
    #  Handle missing values safely
    if pd.isna(text) or text is None:
        return ""
    
    #  Convert to string and lowercase
    text = str(text).casefold()
    
    #  Remove extra whitespace (newlines, tabs, and multiple consecutive spaces)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def preprocess_mcq_data(df, text_columns):
    """
    Applies the clean_text function to all specified columns in the dataframe.
    Returns a new dataframe to avoid mutating the original data.
    """
    df_clean = df.copy()
    
    for col in text_columns:
        df_clean[col] = df_clean[col].apply(clean_text)
        
    return df_clean



# Apply the preprocessing
df_cleaned = preprocess_mcq_data(df, text_cols)

# Verify the changes on the first row
print("Before cleaning:")
print(df.loc[5, 'prompt'])
print("\nAfter cleaning:")
print(df_cleaned.loc[5, 'prompt'])

Before cleaning:
Identify the correct statement: What is the effect generated by a spinning superconductor? among the listed options.

After cleaning:
identify the correct statement: what is the effect generated by a spinning superconductor? among the listed options.


In [13]:
import pandas as pd
options_cols = ['A', 'B', 'C', 'D', 'E']

print("---  PROMPT VS. OPTIONS LENGTH (Word Count) ---")
# Calculate average word count for prompts
avg_prompt_len = df_cleaned['prompt'].str.split().str.len().mean()

# Calculate average word count across all 5 options combined
avg_option_len = df_cleaned[options_cols].apply(lambda x: x.str.split().str.len()).mean().mean()

print(f"Average Prompt length: {avg_prompt_len:.1f} words")
print(f"Average Option length: {avg_option_len:.1f} words")
print(f"Ratio: Prompts are roughly {avg_prompt_len/avg_option_len:.1f}x longer than options.\n")


print("--- 3. VOCABULARY VARIATION (Effect of Lowercasing) ---")
def get_vocab_size(series):
    # Joins all rows into one giant string, splits by space, and counts unique words
    return len(set(' '.join(series.astype(str)).split()))

# Compare raw vocabulary vs lowercased vocabulary on the prompts
raw_vocab_size = get_vocab_size(df['prompt'])
lower_vocab_size = get_vocab_size(df['prompt'].str.lower())
reduction = raw_vocab_size - lower_vocab_size
percent_reduction = (reduction / raw_vocab_size) * 100

print(f"Original unique words: {raw_vocab_size}")
print(f"Lowercased unique words: {lower_vocab_size}")
print(f"Conclusion: Lowercasing reduced the vocabulary size by {reduction} words ({percent_reduction:.1f}%).\n")


print("--- 4. FORMATTING NORMALIZATION (Effect of Whitespace Removal) ---")
# Count total characters before and after cleaning to see how much "invisible" noise was removed
raw_chars = df[text_cols].astype(str).apply(lambda x: x.str.len()).sum().sum()
clean_chars = df_cleaned[text_cols].astype(str).apply(lambda x: x.str.len()).sum().sum()
chars_removed = raw_chars - clean_chars

print(f"Total characters before cleaning: {raw_chars}")
print(f"Total characters after cleaning: {clean_chars}")
print(f"Conclusion: Removed {chars_removed} unnecessary whitespace characters across the dataset.")

---  PROMPT VS. OPTIONS LENGTH (Word Count) ---
Average Prompt length: 18.1 words
Average Option length: 26.3 words
Ratio: Prompts are roughly 0.7x longer than options.

--- 3. VOCABULARY VARIATION (Effect of Lowercasing) ---
Original unique words: 938
Lowercased unique words: 931
Conclusion: Lowercasing reduced the vocabulary size by 7 words (0.7%).

--- 4. FORMATTING NORMALIZATION (Effect of Whitespace Removal) ---
Total characters before cleaning: 1884796
Total characters after cleaning: 1884796
Conclusion: Removed 0 unnecessary whitespace characters across the dataset.


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

corpus = (
    df_cleaned['prompt'] + " " + 
    df_cleaned['A'] + " " + 
    df_cleaned['B'] + " " + 
    df_cleaned['C'] + " " + 
    df_cleaned['D'] + " " + 
    df_cleaned['E']
)

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(corpus)

print("--- VOCABULARY & SHAPE ---")
vocab_size = len(tfidf.vocabulary_)
print(f"Vocabulary Size (Unique words): {vocab_size}")
print(f"Matrix Shape (Documents x Words): {tfidf_matrix.shape}")

print("\n---  SPARSITY ---")
# Calculate how much of the matrix is filled with zeros
total_elements = tfidf_matrix.shape[0] * tfidf_matrix.shape[1]
non_zero_elements = tfidf_matrix.nnz
zero_elements = total_elements - non_zero_elements
sparsity = (zero_elements / total_elements) * 100

print(f"Non-zero elements: {non_zero_elements}")
print(f"Matrix Sparsity: {sparsity:.2f}% zeros")

print("\n---  FEATURE NAMES ---")
feature_names = tfidf.get_feature_names_out()
print(f"First 10 features: {feature_names[:10]}")

print("\n--- 4. EXAMPLE VECTOR (Row 0) ---")
# Extract the first document's vector
row_0 = tfidf_matrix[0]

# .nonzero() returns a tuple of (row_indices, col_indices). We want the columns.
non_zero_indices = row_0.nonzero()[1]

print(f"Words actually present in Document 0: {len(non_zero_indices)}")
print("Top 5 words by TF-IDF importance in Document 0:")

# Map the non-zero indices back to words and their specific TF-IDF scores
doc_0_features = [(feature_names[idx], row_0[0, idx]) for idx in non_zero_indices]

# Sort them by TF-IDF score in descending order to see what the algorithm found most "important"
doc_0_features_sorted = sorted(doc_0_features, key=lambda x: x[1], reverse=True)

for word, score in doc_0_features_sorted[:5]:
    print(f" - {word}: {score:.4f}")

# --- STEP 4: Common vs. Specific Words (Checking the math) ---

# Choose the same sample document (Row 0)
sample_idx = 0
sample_vector = tfidf_matrix[sample_idx]

# Words we want to investigate
common_words = ['the', 'is', 'what']

print("--- 5. COMMON WORD WEIGHTS (Row 0) ---")
for word in common_words:
    # Check if the word exists in our learned vocabulary
    if word in tfidf.vocabulary_:
        # Get the column index for this word
        word_idx = tfidf.vocabulary_[word]
        # Get the TF-IDF score for this word in Document 0
        score = sample_vector[0, word_idx]
        print(f"Score for '{word}': {score:.4f}")
    else:
        print(f"'{word}' not found in vocabulary.")

print("\n--- COMPARISON ---")
print("Notice how the scores for 'the', 'is', and 'what' compare to the top 5 words")
print("we printed earlier in Step 3. Even if the word 'the' appears 5 times in the ")
print("document and a specific word appears only once, the specific word gets a ")
print("higher score because 'the' is penalized for appearing in every other row.")

--- VOCABULARY & SHAPE ---
Vocabulary Size (Unique words): 2940
Matrix Shape (Documents x Words): (2000, 2940)

---  SPARSITY ---
Non-zero elements: 94219
Matrix Sparsity: 98.40% zeros

---  FEATURE NAMES ---
First 10 features: ['000' '013' '013343' '02m' '10' '100' '1012' '1036' '11' '12']

--- 4. EXAMPLE VECTOR (Row 0) ---
Words actually present in Document 0: 90
Top 5 words by TF-IDF importance in Document 0:
 - martin: 0.3382
 - heidegger: 0.3382
 - human: 0.2818
 - future: 0.2818
 - past: 0.2539
--- 5. COMMON WORD WEIGHTS (Row 0) ---
Score for 'the': 0.1852
Score for 'is': 0.0820
Score for 'what': 0.0102

--- COMPARISON ---
Notice how the scores for 'the', 'is', and 'what' compare to the top 5 words
we printed earlier in Step 3. Even if the word 'the' appears 5 times in the 
document and a specific word appears only once, the specific word gets a 
higher score because 'the' is penalized for appearing in every other row.


In [15]:
from sklearn.metrics.pairwise import cosine_similarity

# --- Transform all columns into TF-IDF vectors ---
vec_prompt = tfidf.transform(df_cleaned['prompt'])
vec_A = tfidf.transform(df_cleaned['A'])
vec_B = tfidf.transform(df_cleaned['B'])
vec_C = tfidf.transform(df_cleaned['C'])
vec_D = tfidf.transform(df_cleaned['D'])
vec_E = tfidf.transform(df_cleaned['E'])

# --- Compute Cosine Similarity using sklearn ---
# cosine_similarity returns an NxN matrix. 
# We take .diagonal() to compare row i of prompts only with row i of the options.
sim_A = cosine_similarity(vec_prompt, vec_A).diagonal()
sim_B = cosine_similarity(vec_prompt, vec_B).diagonal()
sim_C = cosine_similarity(vec_prompt, vec_C).diagonal()
sim_D = cosine_similarity(vec_prompt, vec_D).diagonal()
sim_E = cosine_similarity(vec_prompt, vec_E).diagonal()

predictions = []
top_1_predictions = []
all_scores = []
correct_sims = []
wrong_sims_avg = []

for i in range(len(df_cleaned)):
    scores = {
        'A': sim_A[i], 
        'B': sim_B[i], 
        'C': sim_C[i], 
        'D': sim_D[i], 
        'E': sim_E[i]
    }
    
    sorted_opts = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    top_3 = [opt[0] for opt in sorted_opts[:3]]
    
    predictions.append(" ".join(top_3))
    top_1_predictions.append(top_3[0])
    all_scores.append(scores)
    
    # Track similarity for the Correct Answer vs Wrong Answers
    correct_ans = df.iloc[i]['answer']
    correct_sims.append(scores[correct_ans])
    
    # Calculate the mean similarity of the 4 incorrect options
    wrong_scores = [score for opt, score in scores.items() if opt != correct_ans]
    wrong_sims_avg.append(np.mean(wrong_scores))


df_cleaned['Prediction_Top3'] = predictions
df_cleaned['Prediction_Top1'] = top_1_predictions
df_cleaned['Correct_Sim'] = correct_sims
df_cleaned['Wrong_Sim_Avg'] = wrong_sims_avg

In [16]:
# --- EVALUATION METRICS ---
def calculate_map_at_3(predictions_series, actual_series):
    score = 0.0
    for pred_str, actual in zip(predictions_series, actual_series):
        pred_list = pred_str.split()
        if actual in pred_list:
            rank = pred_list.index(actual) + 1
            score += 1.0 / rank # 1 for 1st, 0.5 for 2nd, 0.33 for 3rd
    return score / len(predictions_series)

map3_score = calculate_map_at_3(df_cleaned['Prediction_Top3'], df['answer'])
top1_accuracy = (df_cleaned['Prediction_Top1'] == df['answer']).mean()

print(f"MAP@3 Score: {map3_score:.4f}")
print(f"Top-1 Accuracy: {top1_accuracy:.2%}")

# --- CORRECT VS WRONG SIMILARITY ---
print(f"\nAvg Similarity (Correct Option): {np.mean(correct_sims):.4f}")
print(f"Avg Similarity (Wrong Options):  {np.mean(wrong_sims_avg):.4f}")

# --- DETAILED INSPECTION ---
def print_breakdown(idx, title):
    scores = all_scores[idx]
    sorted_opts = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    correct_ans = df.iloc[idx]['answer']
    
    print(f"\n--- {title} (Row {idx}) ---")
    print(f"Prompt: {df.iloc[idx]['prompt']}")
    print(f"Correct Answer: Option {correct_ans}")
    print(f"Cosine(correct): {scores[correct_ans]:.4f}")
    print(f"Highest cosine:  Option {sorted_opts[0][0]} ({sorted_opts[0][1]:.4f})")
    print(f"Second highest:  Option {sorted_opts[1][0]} ({sorted_opts[1][1]:.4f})")
    print(f"Third highest:   Option {sorted_opts[2][0]} ({sorted_opts[2][1]:.4f})")

# Get indices for successes and failures
success_indices = df_cleaned[df_cleaned['Prediction_Top1'] == df['answer']].index.tolist()
failure_indices = df_cleaned[~df_cleaned.apply(lambda row: row['answer'] in row['Prediction_Top3'].split(), axis=1)].index.tolist()

print("\n" + "="*50)
print("🔍 INSPECTING SUCCESSES")
print("Look for: Definition questions, exact keyword matching, and factual recall.")
for i in success_indices[:3]:  # Print first 3 successes
    print_breakdown(i, "SUCCESS")

print("\n" + "="*50)
print("🔍 INSPECTING FAILURES")
print("Look for: Multi-hop reasoning, synonyms (no keyword overlap), or negation ('Which is NOT...').")
for i in failure_indices[:3]:  # Print first 3 failures
    print_breakdown(i, "FAILURE")

MAP@3 Score: 0.2859
Top-1 Accuracy: 14.40%

Avg Similarity (Correct Option): 0.2967
Avg Similarity (Wrong Options):  0.3138

🔍 INSPECTING SUCCESSES
Look for: Definition questions, exact keyword matching, and factual recall.

--- SUCCESS (Row 5) ---
Prompt: Identify the correct statement: What is the effect generated by a spinning superconductor? among the listed options.
Correct Answer: Option C
Cosine(correct): 0.0214
Highest cosine:  Option C (0.0214)
Second highest:  Option E (0.0212)
Third highest:   Option B (0.0211)

--- SUCCESS (Row 9) ---
Prompt: Which of the following is correct? What is the relationship between the Wigner function and the density matrix operator? from the following choices.
Correct Answer: Option A
Cosine(correct): 0.7222
Highest cosine:  Option A (0.7222)
Second highest:  Option C (0.6413)
Third highest:   Option E (0.6308)

--- SUCCESS (Row 12) ---
Prompt: Choose the correct answer: Which of the following statements accurately describes the relationship bet

In [17]:
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity

# We use a fast, pretrained model for the baseline. 
# You can change this to 'word2vec-google-news-300' for higher accuracy later.
print("Loading pretrained Word2Vec model")
w2v_model = api.load('glove-wiki-gigaword-50') 

def get_sentence_embedding(text, model):
    """
    Converts a sentence into a single dense vector by averaging its word embeddings.
    """
    words = str(text).split()
    # Keep only words that exist in the model's vocabulary
    valid_words = [word for word in words if word in model]
    
    if not valid_words:
        # If no words are found, return a zero vector of the correct dimension
        return np.zeros(model.vector_size)
    
    # Extract vectors and compute the mean along the column axis
    vectors = np.array([model[word] for word in valid_words])
    return vectors.mean(axis=0)

print("Converting text to average word embeddings...")

vec_prompt_w2v = np.vstack(df_cleaned['prompt'].apply(lambda x: get_sentence_embedding(x, w2v_model)))
vec_A_w2v = np.vstack(df_cleaned['A'].apply(lambda x: get_sentence_embedding(x, w2v_model)))
vec_B_w2v = np.vstack(df_cleaned['B'].apply(lambda x: get_sentence_embedding(x, w2v_model)))
vec_C_w2v = np.vstack(df_cleaned['C'].apply(lambda x: get_sentence_embedding(x, w2v_model)))
vec_D_w2v = np.vstack(df_cleaned['D'].apply(lambda x: get_sentence_embedding(x, w2v_model)))
vec_E_w2v = np.vstack(df_cleaned['E'].apply(lambda x: get_sentence_embedding(x, w2v_model)))


sim_A_w2v = cosine_similarity(vec_prompt_w2v, vec_A_w2v).diagonal()
sim_B_w2v = cosine_similarity(vec_prompt_w2v, vec_B_w2v).diagonal()
sim_C_w2v = cosine_similarity(vec_prompt_w2v, vec_C_w2v).diagonal()
sim_D_w2v = cosine_similarity(vec_prompt_w2v, vec_D_w2v).diagonal()
sim_E_w2v = cosine_similarity(vec_prompt_w2v, vec_E_w2v).diagonal()


predictions_w2v = []
top_1_predictions_w2v = []

for i in range(len(df_cleaned)):
    scores = {
        'A': sim_A_w2v[i], 'B': sim_B_w2v[i], 
        'C': sim_C_w2v[i], 'D': sim_D_w2v[i], 'E': sim_E_w2v[i]
    }
    sorted_opts = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    top_3 = [opt[0] for opt in sorted_opts[:3]]
    
    predictions_w2v.append(" ".join(top_3))
    top_1_predictions_w2v.append(top_3[0])

# Add to dataframe for comparison
df_cleaned['Prediction_W2V_Top3'] = predictions_w2v
df_cleaned['Prediction_W2V_Top1'] = top_1_predictions_w2v

# Calculate Metrics
map3_w2v = calculate_map_at_3(df_cleaned['Prediction_W2V_Top3'], df['answer'])
top1_w2v = (df_cleaned['Prediction_W2V_Top1'] == df['answer']).mean()

print("\n--- PHASE 5 EVALUATION ---")
print(f"Word2Vec MAP@3 Score: {map3_w2v:.4f} (vs TF-IDF: {map3_score:.4f})")
print(f"Word2Vec Top-1 Accuracy: {top1_w2v:.2%} (vs TF-IDF: {top1_accuracy:.2%})")

Loading pretrained Word2Vec model
[==================================================] 100.0% 66.0/66.0MB downloaded
Converting text to average word embeddings...

--- PHASE 5 EVALUATION ---
Word2Vec MAP@3 Score: 0.3089 (vs TF-IDF: 0.2859)
Word2Vec Top-1 Accuracy: 12.95% (vs TF-IDF: 14.40%)


In [18]:
# ---  OVERALL COMPARISON ---
# Count how many each got completely right (Top-1)
tfidf_correct_mask = df_cleaned['Prediction_Top1'] == df['answer']
w2v_correct_mask = df_cleaned['Prediction_W2V_Top1'] == df['answer']

print("---  OVERALL WINS ---")
print(f"TF-IDF Correct (Top-1): {tfidf_correct_mask.sum()}")
print(f"Word2Vec Correct (Top-1): {w2v_correct_mask.sum()}")

---  OVERALL WINS ---
TF-IDF Correct (Top-1): 288
Word2Vec Correct (Top-1): 259


In [19]:
# ---  WHICH QUESTIONS IMPROVED? (Word2Vec wins, TF-IDF loses) ---
# Find rows where Word2Vec got Top-1 correct, but TF-IDF didn't even get it in Top-3
w2v_improved_mask = (df_cleaned['Prediction_W2V_Top1'] == df['answer']) & \
                    (~df_cleaned.apply(lambda row: row['answer'] in str(row['Prediction_Top3']).split(), axis=1))

df_improved = df_cleaned[w2v_improved_mask]
print(f"\n---  W2V IMPROVEMENTS ({len(df_improved)} questions) ---")
if not df_improved.empty:
    for i in range(min(2, len(df_improved))): # Print up to 2 examples
        row = df_improved.iloc[i]
        print(f"\nPrompt: {row['prompt']}")
        print(f"Correct Answer ({row['answer']}): {row[row['answer']]}")
        print(f"TF-IDF Guessed: {row['Prediction_Top1']} | Word2Vec Guessed: {row['Prediction_W2V_Top1']}")


---  W2V IMPROVEMENTS (108 questions) ---

Prompt: choose the correct answer: what is the relationship between mass, force, and acceleration, according to sir isaac newton's laws of motion?
Correct Answer (D): mass is an inertial property that determines an object's tendency to remain at constant velocity unless acted upon by an outside force. according to newton's laws of motion and the formula f = ma, an object with a mass of one kilogram accelerates at one meter per second per second when acted upon by a force of one newton.
TF-IDF Guessed: A | Word2Vec Guessed: D

Prompt: choose the correct answer: what is the reason for recycling rare metals according to the united nations?
Correct Answer (E): the demand for rare metals will quickly exceed the consumed tonnage in 2013, and recycling rare metals with a worldwide production lower than 100 000 t/year is urgent and priority should be placed on it in order to conserve natural resources and energy.
TF-IDF Guessed: B | Word2Vec Guessed:

In [20]:
# # ---  WHICH QUESTIONS REGRESSED? (TF-IDF wins, Word2Vec loses) ---
# # Find rows where TF-IDF got Top-1 correct, but Word2Vec failed
# tfidf_wins_mask = (df_cleaned['Prediction_Top1'] == df['answer']) & \
#                   (~df_cleaned.apply(lambda row: row['answer'] in str(row['Prediction_W2V_Top3']).split(), axis=1))

# df_regressed = df_cleaned[tfidf_wins_mask]
# print(f"\n---  TF-IDF WINS ({len(df_regressed)} questions) ---")
# if not df_regressed.empty:
#     for i in range(min(2, len(df_regressed))):
#         row = df_regressed.iloc[i]
#         print(f"\nPrompt: {row['prompt']}")
#         print(f"Correct Answer ({row['answer']}): {row[row['answer']]}")
#         print(f"TF-IDF Guessed: {row['Prediction_Top1']} | Word2Vec Guessed: {row['Prediction_W2V_Top1']}")

# # ---  WHICH QUESTIONS REMAIN DIFFICULT? (Both fail) ---
# both_fail_mask = (~df_cleaned.apply(lambda row: row['answer'] in str(row['Prediction_Top3']).split(), axis=1)) & \
#                  (~df_cleaned.apply(lambda row: row['answer'] in str(row['Prediction_W2V_Top3']).split(), axis=1))

# df_hard = df_cleaned[both_fail_mask]
# print(f"\n---  HARD QUESTIONS ({len(df_hard)} questions) ---")
# print("Look for: Multi-step reasoning, logic puzzles, or 'NOT/Except' questions.")
# if not df_hard.empty:
#     for i in range(min(2, len(df_hard))):
#         row = df_hard.iloc[i]
#         print(f"\nPrompt: {row['prompt']}")
#         print(f"Correct Answer ({row['answer']}): {row[row['answer']]}")
#         print(f"TF-IDF Guessed: {row['Prediction_Top1']} | Word2Vec Guessed: {row['Prediction_W2V_Top1']}")
    

Baseline Transformer 

In [21]:
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm.notebook import tqdm #

In [22]:
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm.notebook import tqdm # For progress bars in Colab

# Set device to GPU if available in your Colab environment
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load Pretrained BERT
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

# VERY IMPORTANT: Set to evaluation mode
model.eval()

Using device: cuda


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [23]:
def get_bert_embeddings(texts, batch_size=32):
    """
    Returns both CLS and Mean embeddings for a list of texts using batching.
    """
    all_cls_embeddings = []
    all_mean_embeddings = []
    
    # Process in batches to avoid running out of Colab GPU memory
    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting Embeddings"):
        batch_texts = texts[i:i+batch_size]
        
        # Tokenize batch
        inputs = tokenizer(
            batch_texts, 
            padding=True, 
            truncation=True, 
            max_length=128, 
            return_tensors="pt"
        ).to(device)
        
        # Inference without tracking gradients
        with torch.no_grad():
            outputs = model(**inputs)
            
            # 1. CLS Embedding: Take the first token [CLS] of the last hidden state
            last_hidden = outputs.last_hidden_state
            cls_embeds = last_hidden[:, 0, :]
            
            # 2. Mean Pooling: Average the token embeddings, ignoring padding tokens
            attention_mask = inputs['attention_mask'].unsqueeze(-1).expand(last_hidden.size()).float()
            sum_embeds = torch.sum(last_hidden * attention_mask, dim=1)
            sum_mask = torch.clamp(attention_mask.sum(dim=1), min=1e-9)
            mean_embeds = sum_embeds / sum_mask
            
            # Move back to CPU and convert to numpy
            all_cls_embeddings.append(cls_embeds.cpu().numpy())
            all_mean_embeddings.append(mean_embeds.cpu().numpy())
            
    return np.vstack(all_cls_embeddings), np.vstack(all_mean_embeddings)

In [24]:
print("Processing Prompts...")
prompt_cls, prompt_mean = get_bert_embeddings(df_cleaned['prompt'].tolist())

print("Processing Option A...")
A_cls, A_mean = get_bert_embeddings(df_cleaned['A'].tolist())

print("Processing Option B...")
B_cls, B_mean = get_bert_embeddings(df_cleaned['B'].tolist())

print("Processing Option C...")
C_cls, C_mean = get_bert_embeddings(df_cleaned['C'].tolist())

print("Processing Option D...")
D_cls, D_mean = get_bert_embeddings(df_cleaned['D'].tolist())

print("Processing Option E...")
E_cls, E_mean = get_bert_embeddings(df_cleaned['E'].tolist())

Processing Prompts...


Extracting Embeddings:   0%|          | 0/63 [00:00<?, ?it/s]

Processing Option A...


Extracting Embeddings:   0%|          | 0/63 [00:00<?, ?it/s]

Processing Option B...


Extracting Embeddings:   0%|          | 0/63 [00:00<?, ?it/s]

... (2 outputs left out)

Processing Option D...


Extracting Embeddings:   0%|          | 0/63 [00:00<?, ?it/s]

Processing Option E...


Extracting Embeddings:   0%|          | 0/63 [00:00<?, ?it/s]

In [25]:
def rank_and_evaluate(vec_prompt, vec_A, vec_B, vec_C, vec_D, vec_E, model_name):
    """Computes similarities, predicts Top-3, and calculates metrics."""
    
    # 1. Compute diagonal cosine similarities
    sim_A = cosine_similarity(vec_prompt, vec_A).diagonal()
    sim_B = cosine_similarity(vec_prompt, vec_B).diagonal()
    sim_C = cosine_similarity(vec_prompt, vec_C).diagonal()
    sim_D = cosine_similarity(vec_prompt, vec_D).diagonal()
    sim_E = cosine_similarity(vec_prompt, vec_E).diagonal()
    
    predictions_top3 = []
    predictions_top1 = []
    
    # 2. Rank for each row
    for i in range(len(df_cleaned)):
        scores = {
            'A': sim_A[i], 'B': sim_B[i], 
            'C': sim_C[i], 'D': sim_D[i], 'E': sim_E[i]
        }
        sorted_opts = sorted(scores.items(), key=lambda item: item[1], reverse=True)
        top_3 = [opt[0] for opt in sorted_opts[:3]]
        
        predictions_top3.append(" ".join(top_3))
        predictions_top1.append(top_3[0])
        
    # 3. Evaluate Metrics
    map3 = calculate_map_at_3(predictions_top3, df['answer'])
    top1 = (pd.Series(predictions_top1) == df['answer']).mean()
    
    print(f"{model_name} -> MAP@3: {map3:.4f} | Top-1: {top1:.2%}")
    return map3, top1

print("--- EVALUATING BERT STRATEGIES ---")
map3_cls, top1_cls = rank_and_evaluate(prompt_cls, A_cls, B_cls, C_cls, D_cls, E_cls, "BERT (CLS)")
map3_mean, top1_mean = rank_and_evaluate(prompt_mean, A_mean, B_mean, C_mean, D_mean, E_mean, "BERT (Mean)")

--- EVALUATING BERT STRATEGIES ---
BERT (CLS) -> MAP@3: 0.3843 | Top-1: 20.80%
BERT (Mean) -> MAP@3: 0.3916 | Top-1: 22.40%


In [26]:
# Create the comparison table
comparison_data = [
    {
        "Model": "TF-IDF", 
        "Embedding": "Sparse Matrix", 
        "Context Awareness": "None (Exact Keyword)",
        "MAP@3": map3_score, 
        "Top-1 Accuracy": top1_accuracy
    },
    {
        "Model": "Word2Vec", 
        "Embedding": "Dense (Static Avg)", 
        "Context Awareness": "Word-level (No Order)",
        "MAP@3": map3_w2v, 
        "Top-1 Accuracy": top1_w2v
    },
    {
        "Model": "BERT (CLS Pooling)", 
        "Embedding": "Dense (Contextual)", 
        "Context Awareness": "Sentence-level",
        "MAP@3": map3_cls, 
        "Top-1 Accuracy": top1_cls
    },
    {
        "Model": "BERT (Mean Pooling)", 
        "Embedding": "Dense (Contextual)", 
        "Context Awareness": "Sentence-level",
        "MAP@3": map3_mean, 
        "Top-1 Accuracy": top1_mean
    }
]

df_comparison = pd.DataFrame(comparison_data)

# Display the table clearly
print("\n--- BASELINE MODELS COMPARISON ---")
print(df_comparison) # Use display() for a nice rendered table in Colab


--- BASELINE MODELS COMPARISON ---
                 Model           Embedding      Context Awareness     MAP@3  \
0               TF-IDF       Sparse Matrix   None (Exact Keyword)  0.285917   
1             Word2Vec  Dense (Static Avg)  Word-level (No Order)  0.308917   
2   BERT (CLS Pooling)  Dense (Contextual)         Sentence-level  0.384250   
3  BERT (Mean Pooling)  Dense (Contextual)         Sentence-level  0.391583   

   Top-1 Accuracy  
0          0.1440  
1          0.1295  
2          0.2080  
3          0.2240  


Submission

In [27]:
# test_cleaned = preprocess_mcq_data(test_df, text_cols)

# # Convert test text to sentence embeddings
# test_prompt_w2v = np.vstack(
#     test_cleaned["prompt"].apply(lambda x: get_sentence_embedding(x, w2v_model))
# )

# test_A_w2v = np.vstack(
#     test_cleaned["A"].apply(lambda x: get_sentence_embedding(x, w2v_model))
# )

# test_B_w2v = np.vstack(
#     test_cleaned["B"].apply(lambda x: get_sentence_embedding(x, w2v_model))
# )

# test_C_w2v = np.vstack(
#     test_cleaned["C"].apply(lambda x: get_sentence_embedding(x, w2v_model))
# )

# test_D_w2v = np.vstack(
#     test_cleaned["D"].apply(lambda x: get_sentence_embedding(x, w2v_model))
# )

# test_E_w2v = np.vstack(
#     test_cleaned["E"].apply(lambda x: get_sentence_embedding(x, w2v_model))
# )


# sim_A = cosine_similarity(test_prompt_w2v, test_A_w2v).diagonal()
# sim_B = cosine_similarity(test_prompt_w2v, test_B_w2v).diagonal()
# sim_C = cosine_similarity(test_prompt_w2v, test_C_w2v).diagonal()
# sim_D = cosine_similarity(test_prompt_w2v, test_D_w2v).diagonal()
# sim_E = cosine_similarity(test_prompt_w2v, test_E_w2v).diagonal()

# predictions = []

# for i in range(len(test_cleaned)):
#     scores = {
#         "A": sim_A[i],
#         "B": sim_B[i],
#         "C": sim_C[i],
#         "D": sim_D[i],
#         "E": sim_E[i],
#     }

#     ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

#     top3 = [label for label, _ in ranked[:3]]

#     predictions.append(" ".join(top3))

In [28]:
# # --- 1. Preprocess Test Data ---
# test_cleaned = preprocess_mcq_data(test_df, text_cols)

# print("Extracting BERT Mean Embeddings for Test Data (this may take a moment)...")

# # --- 2. Generate Embeddings (Batched for GPU) ---
# # We take the second returned value [1] which corresponds to our Mean Pooling outputs
# _, test_prompt_mean = get_bert_embeddings(test_cleaned["prompt"].tolist())
# _, test_A_mean = get_bert_embeddings(test_cleaned["A"].tolist())
# _, test_B_mean = get_bert_embeddings(test_cleaned["B"].tolist())
# _, test_C_mean = get_bert_embeddings(test_cleaned["C"].tolist())
# _, test_D_mean = get_bert_embeddings(test_cleaned["D"].tolist())
# _, test_E_mean = get_bert_embeddings(test_cleaned["E"].tolist())

# # --- 3. Compute Cosine Similarity ---
# sim_A = cosine_similarity(test_prompt_mean, test_A_mean).diagonal()
# sim_B = cosine_similarity(test_prompt_mean, test_B_mean).diagonal()
# sim_C = cosine_similarity(test_prompt_mean, test_C_mean).diagonal()
# sim_D = cosine_similarity(test_prompt_mean, test_D_mean).diagonal()
# sim_E = cosine_similarity(test_prompt_mean, test_E_mean).diagonal()

# # --- 4. Rank Answers ---
# predictions = []

# for i in range(len(test_cleaned)):
#     scores = {
#         "A": sim_A[i],
#         "B": sim_B[i],
#         "C": sim_C[i],
#         "D": sim_D[i],
#         "E": sim_E[i],
#     }

#     # Sort descending
#     ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    
#     # Extract top 3 labels
#     top3 = [label for label, _ in ranked[:3]]
#     predictions.append(" ".join(top3))


In [29]:
# submission = test_df[["id"]].copy()

# submission["Prediction"] = predictions

# submission.head()

# submission.to_csv("submission.csv", index=False)

# print("Submission file saved successfully!")

RAG approach

In [30]:
!pip install faiss-gpu sentence-transformers transformers accelerate datasets

In [31]:
import faiss
from sentence_transformers import SentenceTransformer
from datasets import load_dataset


In [32]:
# import faiss
# from sentence_transformers import SentenceTransformer
# from datasets import load_dataset

# # --- TASK 1: Create a small Knowledge Base ---
# # In reality, load your Wikipedia/STEM corpus here
# knowledge_base = [
#     "The mitochondria is a double-membrane-bound organelle found in most eukaryotic organisms, known as the powerhouse of the cell.",
#     "Photosynthesis is a process used by plants and other organisms to convert light energy into chemical energy.",
#     "Isaac Newton formulated the laws of motion and universal gravitation.",
#     "Linear regression is a linear approach for modelling the relationship between a scalar response and one or more explanatory variables.",
#     "Word2Vec is a group of related models that are used to produce word embeddings, utilizing a two-layer neural network."
# ] # Expand this to 20-100+ documents for your test

# # --- TASK 2: Build FAISS Retriever ---
# print("Loading retrieval model...")
# retriever_model = SentenceTransformer('BAAI/bge-large-en-v1.5')

# print("Encoding Knowledge Base...")
# kb_embeddings = retriever_model.encode(knowledge_base, show_progress_bar=True)

# # Initialize FAISS Index (L2 distance)
# embedding_dim = kb_embeddings.shape[1]
# # Using FlatL2 for exact search. (Use IndexIVFFlat for millions of docs later)
# faiss_index = faiss.IndexFlatL2(embedding_dim) 
# faiss_index.add(kb_embeddings)

# print(f"FAISS index built with {faiss_index.ntotal} documents.")

In [33]:
def chunk_text_with_overlap(text, chunk_size=200, overlap=40, min_chunk_len=30):
    """
    Splits text into fixed-size word chunks with overlap to maintain semantic continuity.
    """
    words = text.split()
    if len(words) <= chunk_size:
        return [" ".join(words)] if len(words) >= min_chunk_len else []
    
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunk = words[i : i + chunk_size]
        if len(chunk) >= min_chunk_len:
            chunks.append(" ".join(chunk))
    return chunks

In [34]:
# --- STEP 2: Load Wikipedia & Build Knowledge Base ---
print("Connecting to Hugging Face Wikipedia database...")
wiki_dataset = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)

print("Building Knowledge Base with fixed-size overlapping chunks...")
knowledge_base = []
NUM_ARTICLES = 10000  # Adjust as needed

for i, article in enumerate(tqdm(wiki_dataset, total=NUM_ARTICLES, desc="Processing Articles")):
    if i >= NUM_ARTICLES:
        break
    
    # Generate overlapping chunks per article
    article_chunks = chunk_text_with_overlap(article['text'], chunk_size=200, overlap=40)
    knowledge_base.extend(article_chunks)

print(f"Successfully created {len(knowledge_base):,} overlapping chunks!")

Connecting to Hugging Face Wikipedia database...


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Building Knowledge Base with fixed-size overlapping chunks...


Processing Articles:   0%|          | 0/10000 [00:00<?, ?it/s]

Successfully created 94,018 overlapping chunks!


In [35]:
# --- STEP 3: Switch Retriever Model ---
# BAAI/bge-base-en-v1.5 is significantly stronger than all-MiniLM-L6-v2
retriever_model_name = 'BAAI/bge-base-en-v1.5'
print(f"Loading retriever model: {retriever_model_name}...")
retriever_model = SentenceTransformer(retriever_model_name)

print("Encoding Knowledge Base (this takes a few minutes)...")
kb_embeddings = retriever_model.encode(
    knowledge_base, 
    batch_size=256, 
    show_progress_bar=True, 
    normalize_embeddings=True # Normalizing makes IndexFlatIP equivalent to Cosine Similarity
)

# Initialize FAISS Index using Inner Product (Cosine Similarity)
embedding_dim = kb_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(np.array(kb_embeddings).astype('float32'))

print(f"FAISS index ready with {faiss_index.ntotal:,} vectors.")

Loading retriever model: BAAI/bge-base-en-v1.5...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding Knowledge Base (this takes a few minutes)...


Batches:   0%|          | 0/368 [00:00<?, ?it/s]

FAISS index ready with 94,018 vectors.


In [ ]:
# import faiss
# import numpy as np
# from sentence_transformers import SentenceTransformer

# # --- STEP 3: Switch Retriever Model (Dual GPU) ---
# retriever_model_name = 'BAAI/bge-base-en-v1.5'
# print(f"Loading retriever model: {retriever_model_name}...")
# retriever_model = SentenceTransformer(retriever_model_name)

# print("Spinning up multi-process pool for dual GPUs...")
# # This automatically detects both T4 GPUs and spawns a worker for each
# pool = retriever_model.start_multi_process_pool()

# print("Encoding Knowledge Base across both T4 GPUs...")
# # Distribute the encoding workload across the workers
# kb_embeddings = retriever_model.encode_multi_process(
#     knowledge_base, 
#     pool, 
#     batch_size=256
# )

# print("Shutting down GPU workers...")
# retriever_model.stop_multi_process_pool(pool)

# # Note: encode_multi_process does not accept the normalize_embeddings flag directly.
# # We manually normalize the resulting vectors in-place using FAISS so IndexFlatIP 
# # correctly behaves as Cosine Similarity.
# faiss.normalize_L2(kb_embeddings)

# # Initialize FAISS Index using Inner Product (Cosine Similarity)
# embedding_dim = kb_embeddings.shape[1]
# faiss_index = faiss.IndexFlatIP(embedding_dim)
# faiss_index.add(kb_embeddings.astype('float32'))

# print(f"FAISS index ready with {faiss_index.ntotal:,} vectors.")

Loading retriever model: BAAI/bge-base-en-v1.5...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Spinning up multi-process pool for dual GPUs...
Encoding Knowledge Base across both T4 GPUs...


/tmp/ipykernel_57/1587079042.py:16: DeprecationWarning: The `encode_multi_process` method has been deprecated, and its functionality has been integrated into `encode`. You can now call `encode` with the same parameters to achieve multi-process encoding.
  kb_embeddings = retriever_model.encode_multi_process(


In [42]:
# # --- TASK 3: Retrieve Top 3 documents ---
# top_k = 3
# print("Retrieving context for prompts...")

# # Encode the prompts from your cleaned dataframe
# prompt_embeddings = retriever_model.encode(df_cleaned['prompt'].tolist(), show_progress_bar=True)

# # Search FAISS
# distances, indices = faiss_index.search(prompt_embeddings, top_k)

# # Map indices back to the actual text
# retrieved_contexts = []
# for idx_list in indices:
#     # Combine the top 3 retrieved chunks into one large context string
#     context_str = " ".join([knowledge_base[i] for i in idx_list if i < len(knowledge_base)])
#     retrieved_contexts.append(context_str)

# df_cleaned['RAG_Context'] = retrieved_contexts

Retrieving context for prompts...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

In [36]:
# --- STEP 4: Query Reformulation ---
def construct_reformulated_query(row):
    """
    Embeds Question + first sentence/summary of each option so FAISS can search for 
    specific key terms contained in the options as well as the prompt.
    """
    prompt = str(row['prompt']).strip()
    
    # Extract first sentence of each option to prevent overly long query noise
    opt_a = str(row['A']).split('.')[0]
    opt_b = str(row['B']).split('.')[0]
    opt_c = str(row['C']).split('.')[0]
    opt_d = str(row['D']).split('.')[0]
    opt_e = str(row['E']).split('.')[0]
    
    reformulated = (
        f"{prompt} | Key Options: "
        f"A: {opt_a}; B: {opt_b}; C: {opt_c}; D: {opt_d}; E: {opt_e}"
    )
    return reformulated

print("Reformulating search queries...")
df_cleaned['Reformulated_Query'] = df_cleaned.apply(construct_reformulated_query, axis=1)

# --- STEP 5: Perform FAISS Search ---
TOP_K = 5  # Experiment with 3, 5, or 10

print(f"Encoding reformulated queries and searching Top-{TOP_K} documents...")
query_embeddings = retriever_model.encode(
    df_cleaned['Reformulated_Query'].tolist(), 
    show_progress_bar=True, 
    normalize_embeddings=True
)

# Search FAISS
distances, indices = faiss_index.search(np.array(query_embeddings).astype('float32'), TOP_K)

# Store retrieved lists in dataframe
all_retrieved_contexts = []
for idx_list in indices:
    chunks_for_row = [knowledge_base[i] for i in idx_list if i < len(knowledge_base)]
    all_retrieved_contexts.append(chunks_for_row)

df_cleaned['Retrieved_Chunks'] = all_retrieved_contexts
# Combine chunks into one context string for LLM input
df_cleaned['RAG_Context'] = df_cleaned['Retrieved_Chunks'].apply(lambda chunks: "\n---\n".join(chunks))

Reformulating search queries...
Encoding reformulated queries and searching Top-5 documents...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

In [37]:
def inspect_retrieval_quality(df_data, sample_indices=[0, 1, 2]):
    """
    Prints Question, Options, and Top Retrieved Paragraphs for visual verification.
    """
    for idx in sample_indices:
        row = df_data.iloc[idx]
        print("=" * 80)
        print(f" QUESTION (Row {idx}): {row['prompt']}")
        print(f" CORRECT ANSWER: Option {row['answer']}")
        print("-" * 80)
        print(" OPTIONS:")
        print(f"  A) {row['A']}")
        print(f"  B) {row['B']}")
        print(f"  C) {row['C']}")
        print(f"  D) {row['D']}")
        print(f"  E) {row['E']}")
        print("-" * 80)
        print(f" REFORMULATED QUERY USED FOR RETRIEVAL:\n  {row['Reformulated_Query']}")
        print("-" * 80)
        
        chunks = row['Retrieved_Chunks']
        for rank, chunk in enumerate(chunks, start=1):
            score = distances[idx][rank-1]
            print(f"\n [Retrieved Paragraph {rank} | Similarity Score: {score:.4f}]")
            print(f" {chunk}")
        print("=" * 80 + "\n")

# Run inspection on the first 3 questions
inspect_retrieval_quality(df_cleaned, sample_indices=[0, 1, 2])

 QUESTION (Row 0): pick the best possible answer: what is martin heidegger's view on the relationship between time and human existence? among the listed options.
 CORRECT ANSWER: Option B
--------------------------------------------------------------------------------
 OPTIONS:
  A) martin heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. the relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.
  B) martin heidegger believes that humans do not exist inside time, but that they are time. the relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
  C) martin heidegger does not believe in the existence of time or that it has any effect on human consciousness. the relationship to the past a

In [38]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# --- TASK 4 & 5: Setup LLM ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Loading Generative LLM on {device}...")

llm_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(llm_name).to(device)
llm.eval()

# Pre-compute token IDs for our target letters to extract probabilities later
# T5 uses specific token IDs for 'A', 'B', 'C', 'D', 'E'
target_tokens = ['A', 'B', 'C', 'D', 'E']
target_ids = [tokenizer.encode(t, add_special_tokens=False)[0] for t in target_tokens]

def format_rag_prompt(row):
    """Constructs the prompt containing the retrieved knowledge and the question."""
    prompt = f"Context: {row['RAG_Context']}\n\n"
    prompt += f"Question: {row['prompt']}\n"
    prompt += f"A) {row['A']}\n"
    prompt += f"B) {row['B']}\n"
    prompt += f"C) {row['C']}\n"
    prompt += f"D) {row['D']}\n"
    prompt += f"E) {row['E']}\n\n"
    prompt += "Answer with only the letter of the correct option (A, B, C, D, or E)."
    return prompt

df_cleaned['RAG_Prompt'] = df_cleaned.apply(format_rag_prompt, axis=1)

Loading Generative LLM on cuda...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [39]:
# --- TASK 6: Collect Predictions and Rank ---
rag_predictions_top3 = []
rag_predictions_top1 = []

print("Running LLM Inference...")
for prompt_text in tqdm(df_cleaned['RAG_Prompt'].tolist()):
    
    inputs = tokenizer(prompt_text, return_tensors="pt", max_length=512, truncation=True).to(device)
    
    # For T5, we feed a start token to the decoder to get the first predicted token
    decoder_input_ids = tokenizer("<pad>", return_tensors="pt").input_ids.to(device)
    
    with torch.no_grad():
        outputs = llm(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask, decoder_input_ids=decoder_input_ids)
        
        # Get the logits for the very first generated token
        next_token_logits = outputs.logits[0, 0, :]
        
        # Extract the logits specifically for A, B, C, D, E
        letter_logits = [next_token_logits[tid].item() for tid in target_ids]
        
    # Map letters to their scores
    scores = {letter: score for letter, score in zip(target_tokens, letter_logits)}
    
    # Rank descending based on the LLM's internal confidence
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    top3 = [item[0] for item in ranked[:3]]
    
    rag_predictions_top3.append(" ".join(top3))
    rag_predictions_top1.append(top3[0])

df_cleaned['Prediction_RAG_Top3'] = rag_predictions_top3
df_cleaned['Prediction_RAG_Top1'] = rag_predictions_top1

Running LLM Inference...


  0%|          | 0/2000 [00:00<?, ?it/s]

In [40]:
# --- TASK 7: Compare Against Baseline ---
map3_rag = calculate_map_at_3(df_cleaned['Prediction_RAG_Top3'], df['answer'])
top1_rag = (df_cleaned['Prediction_RAG_Top1'] == df['answer']).mean()

print("\n--- MILESTONE 3: RAG EVALUATION ---")
print(f"RAG MAP@3 Score: {map3_rag:.4f}")
print(f"RAG Top-1 Accuracy: {top1_rag:.2%}")


--- MILESTONE 3: RAG EVALUATION ---
RAG MAP@3 Score: 0.3527
RAG Top-1 Accuracy: 16.70%


In [46]:
rag_data = {
    "Model": "Flan-T5 + FAISS (RAG)", 
    "Embedding": "Generative + Retrieval", 
    "Context Awareness": "Open-Book Reasoning",
    "MAP@3": map3_rag, 
    "Top-1 Accuracy": top1_rag
}

# Append to df_comparison if you have it in memory
df_comparison = pd.concat([df_comparison, pd.DataFrame([rag_data])], ignore_index=True)
display(df_comparison)

,Model,Embedding,Context Awareness,MAP@3,Top-1 Accuracy
0,TF-IDF,Sparse Matrix,None (Exact Keyword),0.285917,0.1440
1,Word2Vec,Dense (Static Avg),Word-level (No Order),0.308917,0.1295
2,BERT (CLS Pooling),Dense (Contextual),Sentence-level,0.384250,0.2080
3,BERT (Mean Pooling),Dense (Contextual),Sentence-level,0.391583,0.2240
4,Flan-T5 + FAISS (RAG),Generative + Retrieval,Open-Book Reasoning,0.543333,0.3590
5,Flan-T5 + FAISS (RAG),Generative + Retrieval,Open-Book Reasoning,0.373500,0.2180


In [49]:
# list(df["prompt"].head(100))